# Momentum Transformer Pipeline
This notebook executes the end-to-end pipeline described in the paper *Trading with the Momentum Transformer*.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import torch
import numpy as np
from data import download_data
from features import compute_features
from dataset import get_dataloaders
from model import MomentumTransformer
from train import train_model
from backtest import run_backtest

## 2. Data Download
Downloading OHLCV data for a basket of liquid ETFs representing different asset classes.

In [ ]:
tickers = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'IEF', 'SHY', 'LQD', 'HYG', 'GLD', 'SLV', 'USO', 'UNG', 'DBA', 'VNQ', 'XLF', 'XLK', 'XLV', 'XLE', 'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'NVDA', 'TSLA', 'BRK-B', 'UNH', 'JNJ', 'JPM', 'V', 'PG', 'MA', 'HD', 'CVX', 'MRK', 'ABBV', 'LLY', 'PEP', 'KO', 'BAC', 'AVGO', 'TMO', 'COST', 'MCD', 'CSCO', 'WMT', 'CRM', 'DIS', 'PFE', 'DHR', 'NKE', 'XOM', 'TXN', 'UNP', 'LIN', 'ADBE', 'VZ', 'PM', 'NEE', 'RTX', 'QCOM', 'INTC', 'AMD', 'HON', 'UPS', 'AMGN', 'IBM', 'BA']
start_date = '2005-01-01'
end_date = '2024-01-01'
df_raw = download_data(tickers, start_date, end_date, 'data/raw_data.csv')
df_raw.head()

## 3. Feature Engineering
Computing multi-horizon returns, volatility scaling, and MACD indicators.

In [ ]:
df_features = compute_features(df_raw)
df_features.to_csv('data/features.csv', index=False)
df_features.tail()

## 4. Leakage-safe Train/Val/Test Split & Dataloaders

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(df_features, batch_size=256)
print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

## 5 & 6. Model Architecture & Training Loop
Training the Decoder-Only TFT to maximize the Sharpe ratio.

In [ ]:
# This will train the model and save the best weights to models/best_model.pth
# Note: This might take some time depending on your hardware.
train_model()

## 7 & 8. Backtesting & Metrics Computation
Running the backtest with transaction costs and calculating all requested metrics.

In [ ]:
metrics, bench_metrics, port_returns, bench_returns = run_backtest('data/features.csv', 'models/best_model.pth', cost_bps=3)

## 9. Dashboard
To view the interactive dashboard with equity curves, drawdowns, and interpretability insights, run the following command in your terminal:
`streamlit run dashboard.py`